In [45]:
import pandas as pd
import numpy as np
import json
import random as rnd
import string
import datetime
from dateutil.relativedelta import relativedelta
import zipfile
import os

Create Empty Data Frames for All CSVs We want.

In [257]:
leads = pd.DataFrame()
opportunities = pd.DataFrame()
accounts = pd.DataFrame()
users = pd.DataFrame()
campaigns = pd.DataFrame()
campaign_members = pd.DataFrame()
last_trigger = {"Content" : datetime.date(2025, 8, 13), "Event": datetime.date(2025, 8, 1)}

In [332]:
#Function to return the probability of an event happening in a given day
def day_prob(pMin, pMax, dMin, dMax):
    """Simulate random day probabilities."""
    return 1.0 - (1.0 - rnd.uniform(pMin, pMax)) ** (1.0 / rnd.randint(dMin, dMax))


def generate_random_deals(pMin,pMax, growth):
    """Simulate random partnership/referral deals."""
    count = 0
    while rnd.uniform(0, 1) < (rnd.uniform(pMin,pMax) * growth):
        count += 1
    return count


def id_writer(type, number):

    with open("Rules_Tables/Tracker_ID.json", "r", encoding="utf-8") as file:
        Tracker_ID = json.load(file)
    
    current_ID = int(Tracker_ID[type][-10:])
    new_ids = []

    for id in range(number):
        current_ID += 1
        new_ids.append(f"ID{current_ID:010d}")

    Tracker_ID[type] = f"ID{current_ID:010d}"

    with open("Rules_Tables/Tracker_ID.json", "w", encoding="utf-8") as file:
        json.dump(Tracker_ID, file)
    
    return new_ids


In [48]:
lead_source_rules = pd.read_csv("Rules_Tables/Lead_Source.csv")


,Name,Min,Max
0,Sales,0.05,0.10
1,Organic,0.15,0.25
2,Event,0.10,0.20
3,Content,0.10,0.20
4,Partnership,0.25,0.40
5,Referral,0.25,0.40


In [49]:
lead_source_rules['Randomizer'] = lead_source_rules.apply(lambda x: day_prob(x['Min'], x['Max'], 30, 100), axis = 1)
lead_source_rules

,Name,Min,Max,Randomizer
0,Sales,0.05,0.10,0.001241
1,Organic,0.15,0.25,0.002148
2,Event,0.10,0.20,0.002272
3,Content,0.10,0.20,0.001548
4,Partnership,0.25,0.40,0.008672
5,Referral,0.25,0.40,0.003182


In [345]:
CONTENT_DECAY_DAYS = 5
EVENT_MIN_DAYS = 30
EVENT_SCALE = 120

def generate_daily_leads(today, growth, last_trigger):
    """Generate today's new leads by category."""
    leads = {"Organic": 0, "Content": 0, "Event": 0,
             "Partnership": 0, "Referral": 0}

    last_content_trigger = today - last_trigger["Content"]
    last_event_trigger = today - last_trigger["Event"]

    if today.weekday() < 5:  # Weekday
        leads["Organic"] = int(round(rnd.randint(15, 25) * growth, 0))

        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            leads["Content"] = int(round(
                (rnd.randint(10, 30) * growth) - (2 * last_content_trigger.days), 0
            ))

        if rnd.uniform(0, 1) <= day_prob(0.25, 0.50, 30, 30):
            leads["Content"] = int(round(rnd.randint(10, 30) * growth, 0))
            last_trigger["Content"] = today

        if (
            rnd.uniform(0, 1)
            < ((last_event_trigger.days / EVENT_SCALE) * growth)
            and last_event_trigger.days > EVENT_MIN_DAYS
            ):
            leads["Event"] = int(round(rnd.randint(0, 200) * growth, 0))

        leads["Partnership"] = generate_random_deals(0.05, 0.1, growth)
        leads["Referral"] = generate_random_deals(0.05, 0.1, growth)

    else:  # Weekend
        leads["Organic"] = int(round(rnd.randint(5, 15) * growth, 0))
        if last_content_trigger.days <= CONTENT_DECAY_DAYS:
            leads["Content"] = int(round(rnd.randint(2, 5) * growth, 0))

    lead_list = []
    
    for type, value in leads.items():    
        #print(f"{type} = {value}")
        lead_list = lead_list + ([type] * value )
    
    lead_count = len(lead_list)
    rnd.shuffle(lead_list)

    new_leads = pd.DataFrame({
        "Lead ID" : id_writer("Lead", lead_count),
        "Create Date": [today] * lead_count,
        "Lead Status": ['Marketing Qualified'] * lead_count,
        "Lead Source": lead_list})
    
    return new_leads


In [356]:
TODAY = datetime.date(2025, 8, 25)
GROWTH = 2.0
generated_leads = generate_daily_leads(TODAY, GROWTH, last_trigger)
generated_leads.head(5)

ID0000000832


,Lead Source,Lead Status,Create Date,Lead ID
0,Organic,Marketing Qualified,2025-08-25,ID0000000757
1,Organic,Marketing Qualified,2025-08-25,ID0000000758
2,Content,Marketing Qualified,2025-08-25,ID0000000759
3,Organic,Marketing Qualified,2025-08-25,ID0000000760
4,Content,Marketing Qualified,2025-08-25,ID0000000761
